# 01 · Data Preparation — Filtering Danish AIS Data to Fishing Vessels

**Team Vega · SODAS Hackathon 2026 · University of Copenhagen**

The raw [AIS](https://en.wikipedia.org/wiki/Automatic_identification_system) (Automatic
Identification System) datasets published by the Danish Maritime Authority contain **every
vessel broadcast** around Denmark — well over 100 million rows per year. Loading that into
memory on a laptop is not an option.

This notebook does the first, foundational step of the project: it **streams** each yearly
file with [Polars](https://pola.rs/) lazy scanning and writes out a much smaller, per-year
file containing **only fishing vessels**. Every later analysis builds on these slimmed-down
files.

**Input:**  `../data/raw/aisdk-<year>-1h.parquet` (raw AIS — download separately, see the README)
**Output:** `../data/fishing/fishing_aisdk-<year>-1h.parquet`

> **Why lazy scanning?** `pl.scan_parquet()` builds a *query plan* without loading anything.
> Polars only reads the rows and columns it actually needs when we call `.collect()`, so we
> can filter 100M+ rows on a normal laptop. See `docs/BIG_DATA_TIPS.md`.

In [ ]:
import os

import polars as pl

# Keep Polars from grabbing every CPU core (and freezing the laptop) during the hackathon.
os.environ["POLARS_MAX_THREADS"] = "4"

# All paths are relative to this notebook (which lives in notebooks/).
RAW_DIR = "../data/raw"
FISHING_DIR = "../data/fishing"
os.makedirs(FISHING_DIR, exist_ok=True)

YEARS = [2021, 2022, 2023, 2024, 2025, 2026]

## 1. Inspect the raw schema

Before filtering, we peek at the schema and a small sample so we know the exact column
names (the AIS feed uses quirky names like `# Timestamp` and `Navigational status`).

In [ ]:
df_lazy = pl.scan_parquet(f"{RAW_DIR}/aisdk-2021-1h.parquet")
df_lazy.collect_schema()

In [ ]:
# Pull only the first 10k rows so we can eyeball the data without loading the full file.
sample = df_lazy.head(10_000).collect()
sample.head()

In [ ]:
# What vessel categories and navigational statuses exist?
print(sample["Ship type"].unique().to_list())
print(sample["Navigational status"].unique().to_list())

## 2. Filter each year to fishing vessels

We keep only rows where `Ship type == "Fishing"` and write one slim Parquet file per year.
This is the single most important data-reduction step in the project — it turns ~100M-row
national files into a few-million-row fishing dataset we can actually work with.

In [ ]:
for year in YEARS:
    in_path = f"{RAW_DIR}/aisdk-{year}-1h.parquet"
    out_path = f"{FISHING_DIR}/fishing_aisdk-{year}-1h.parquet"

    fishing = (
        pl.scan_parquet(in_path)
        .filter(pl.col("Ship type") == "Fishing")
        .collect()  # add streaming=True here if the file is too big for RAM
    )
    fishing.write_parquet(out_path)
    print(f"{year}: {fishing.height:,} fishing rows -> {out_path}")

## 3. Quick sanity check

Re-open one of the saved files and confirm it only contains fishing vessels and the columns
we expect downstream.

In [ ]:
check = pl.read_parquet(f"{FISHING_DIR}/fishing_aisdk-2022-1h.parquet")
print("rows:", f"{check.height:,}")
print("ship types:", check["Ship type"].unique().to_list())
check.head()

---
**Next:** [`02_fishing_heatmaps.ipynb`](02_fishing_heatmaps.ipynb) turns these files into
year-by-year heatmaps of fishing activity around Denmark.